In [ ]:

from pathlib import Path
import re
import time
import zipfile

import requests
from tqdm import tqdm

# ========== 配置 ==========
DATASET_ID = "nm000290"
VERSION = "v1.0.0"

SAVE_DIR = Path(r"F:\ShengbinDuan\NIPS2026\dataset1")

URL = (
    f"https://data.nemar.org/"
    f"{DATASET_ID}/{VERSION}.zip"
)

ZIP_PATH = SAVE_DIR / f"{DATASET_ID}_{VERSION}.zip"
PART_PATH = SAVE_DIR / f"{DATASET_ID}_{VERSION}.zip.part"

CHUNK_SIZE = 1024 * 1024   # 1 MB
MAX_RETRIES = 6


def download():
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    if ZIP_PATH.exists():
        print(f"文件已经存在：{ZIP_PATH}")
        print("如需重新下载，请先删除该 ZIP 文件。")
        return

    print("=" * 60)
    print("Dataset: PROTEUS BCI Bordeaux")
    print(f"URL: {URL}")
    print(f"Save to: {ZIP_PATH}")
    print("=" * 60)

    for attempt in range(1, MAX_RETRIES + 1):
        existing = PART_PATH.stat().st_size if PART_PATH.exists() else 0

        headers = {"Accept-Encoding": "identity"}
        if existing:
            headers["Range"] = f"bytes={existing}-"
            print(f"\n断点续传：{existing / 1024**2:.2f} MB")

        try:
            with requests.get(
                URL,
                headers=headers,
                stream=True,
                timeout=(20, 120),
                allow_redirects=True,
            ) as response:

                # 已经下载完，但还没重命名
                if response.status_code == 416:
                    match = re.search(
                        r"\*/(\d+)",
                        response.headers.get("Content-Range", "")
                    )
                    if match and int(match.group(1)) == existing:
                        PART_PATH.replace(ZIP_PATH)
                        break

                    PART_PATH.unlink(missing_ok=True)
                    raise RuntimeError("断点无效，下一次将重新下载")

                response.raise_for_status()

                # 确认服务端是否支持续传
                resume = existing > 0 and response.status_code == 206

                if resume:
                    content_range = response.headers.get("Content-Range", "")
                    match = re.match(
                        r"bytes (\d+)-\d+/(\d+|\*)", content_range
                    )
                    if not match or int(match.group(1)) != existing:
                        raise RuntimeError("服务器返回的续传区间不正确")

                    total = (
                        int(match.group(2))
                        if match.group(2) != "*"
                        else None
                    )
                else:
                    if existing:
                        print("服务器未接受续传，重新下载整个文件。")
                    existing = 0
                    content_length = response.headers.get("Content-Length")
                    total = int(content_length) if content_length else None

                mode = "ab" if resume else "wb"

                with open(PART_PATH, mode) as f:
                    with tqdm(
                        total=total,
                        initial=existing,
                        unit="B",
                        unit_scale=True,
                        unit_divisor=1024,
                        desc="Downloading",
                        dynamic_ncols=True,
                    ) as bar:

                        for chunk in response.iter_content(
                            chunk_size=CHUNK_SIZE
                        ):
                            if chunk:
                                f.write(chunk)
                                bar.update(len(chunk))

                # 确保文件没有提前结束
                if total is not None:
                    actual = PART_PATH.stat().st_size
                    if actual != total:
                        raise IOError(
                            f"下载不完整：{actual}/{total} bytes"
                        )

                PART_PATH.replace(ZIP_PATH)
                break

        except (requests.RequestException, OSError, RuntimeError) as e:
            print(f"\n第 {attempt} 次尝试失败：{e}")

            if attempt == MAX_RETRIES:
                raise RuntimeError(
                    "下载失败，请检查网络后重新运行脚本。"
                    "已下载的数据保留在 .part 文件中。"
                ) from e

            time.sleep(min(attempt * 3, 20))

    # 检查 ZIP 基本结构
    print("\n检查 ZIP 文件结构...")
    try:
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            print(f"ZIP 结构正常，包含 {len(zf.infolist())} 个文件")
    except zipfile.BadZipFile as e:
        raise RuntimeError(
            "ZIP 文件结构损坏，请删除该 ZIP 后重新下载。"
        ) from e

    size_gb = ZIP_PATH.stat().st_size / (1024 ** 3)

    print("\n" + "=" * 60)
    print("下载完成！")
    print(f"文件位置：{ZIP_PATH}")
    print(f"文件大小：{size_gb:.3f} GiB")
    print("=" * 60)


if __name__ == "__main__":
    download()


In [ ]:
from pathlib import Path
import zipfile

root = Path(r"F:\ShengbinDuan\NIPS2026\dataset")

zip_path = root / "nm000290_v1.0.0.zip"
extract_path = root / "nm000290_v1.0.0"

extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    for info in tqdm(zf.infolist(), desc="Extracting", unit="file"):
        zf.extract(info, extract_path)

print("解压完成：", extract_path)